# 4 · Reporting — *Putting it in front of the board*

> **Chapter 4 of the Aurora story.** The analysis is done; now it has to land with people who don't read
> trace plots. `mmm_framework` renders a **self-contained, themed HTML report** straight from a fitted
> model — executive summary, ROAS with uncertainty, decomposition, diagnostics, and a causal-assumptions
> section so stakeholders see what the number rests on.

In [1]:
import warnings, sys
warnings.filterwarnings("ignore")
from loguru import logger
logger.remove(); logger.add(sys.stderr, level="ERROR")   # quiet framework logs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aurora import generate_aurora, CHANNELS, PRODUCTS, PALETTE, CHANNEL_COLORS

plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (9, 4.2),
    "axes.grid": True, "grid.alpha": 0.18,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#cfc7bd", "axes.titleweight": "bold",
    "figure.facecolor": "white", "savefig.facecolor": "white", "font.size": 10,
})
ACCENT, INK, MUTED = PALETTE["accent"], PALETTE["ink"], PALETTE["muted"]

aurora = generate_aurora()      # the one dataset every chapter shares

## Fit (or reuse) the Aurora model, then generate the report

The report builds directly from a fitted `BayesianMMM` — it extracts contributions, ROAS, fit
statistics and diagnostics for you.

In [2]:
from mmm_framework import BayesianMMM, ModelConfigBuilder, SeasonalityConfigBuilder, TrendConfig, TrendType

panel = aurora.base_panel(control_demand=True)
model_config = (ModelConfigBuilder().bayesian_pymc().with_chains(2).with_draws(400).with_tune(400)
                .with_seasonality_builder(SeasonalityConfigBuilder().with_yearly(order=2)).build())
mmm = BayesianMMM(panel, model_config, TrendConfig(type=TrendType.LINEAR))
results = mmm.fit(draws=400, tune=400, chains=2, cores=1, random_seed=0)

Sampling: [adstock_alpha_Display, adstock_alpha_Search, adstock_alpha_Social, adstock_alpha_TV, beta_Display, beta_Search, beta_Social, beta_TV, beta_controls, intercept, sat_lam_Display, sat_lam_Search, sat_lam_Social, sat_lam_TV, season_yearly, sigma, trend_slope, y_obs]


Initializing NUTS using adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [intercept, trend_slope, season_yearly, adstock_alpha_TV, sat_lam_TV, beta_TV, adstock_alpha_Search, sat_lam_Search, beta_Search, adstock_alpha_Social, sat_lam_Social, beta_Social, adstock_alpha_Display, sat_lam_Display, beta_Display, beta_controls, sigma]


Output()

Sampling 2 chains for 400 tune and 400 draw iterations (800 + 800 draws total) took 8 seconds.


There were 6 divergences after tuning. Increase `target_accept` or reparameterize.


We recommend running at least 4 chains for robust computation of convergence diagnostics


In [3]:
from pathlib import Path
from mmm_framework.reporting import (
    MMMReportGenerator, ReportConfig, SectionConfig, ColorScheme, ColorPalette)

Path("artifacts").mkdir(exist_ok=True)
config = ReportConfig(
    title="Aurora Coffee Co. — Marketing Effectiveness",
    subtitle="Q3 Planning Review",
    client="Aurora Coffee Co.",
    analysis_period="2023 – 2024 (weekly)",
    color_scheme=ColorScheme.from_palette(ColorPalette.CORPORATE),
    confidential=True,
    diagnostics=SectionConfig(enabled=True),
)
report = MMMReportGenerator(model=mmm, panel=panel, results=results, config=config)
out = report.to_html("artifacts/aurora_report.html")
print(f"wrote {out}  ({len(report.render()):,} chars of HTML)")

Sampling: [y_obs]


Output()

Sampling: [y_obs]


Output()

Sampling: [y_obs]


...

Sampling: [y_obs]


Output()

Sampling: [y_obs]


Output()

wrote artifacts/aurora_report.html  (1,050,457 chars of HTML)


> **Note:** charts use Plotly loaded from a CDN, so rendering needs a network connection. The saved file
> is otherwise fully self-contained.

### Preview it inline

In [4]:
from IPython.display import IFrame
IFrame("artifacts/aurora_report.html", width="100%", height=560)

## Theming & sections

Swap palettes, toggle sections, and brand the deck. There are four built-in palettes and a fluent
`ReportBuilder` for one-liners.

In [5]:
from mmm_framework.reporting import ReportBuilder

# Executive cut: only the sections a CMO wants, in the warm palette.
exec_report = (ReportBuilder()
    .with_model(mmm, panel=panel, results=results)
    .with_title("Aurora — Executive Summary").with_client("Aurora Coffee Co.")
    .enable_all_sections().disable_section("methodology").disable_section("diagnostics")
    .with_credible_interval(0.8)
    .build())
exec_report.to_html("artifacts/aurora_exec_summary.html")

print("Built-in palettes:", [p.name for p in ColorPalette])
print("Saved: artifacts/aurora_report.html (full) + artifacts/aurora_exec_summary.html (exec cut)")

Sampling: [y_obs]


Output()

Sampling: [y_obs]


Output()

Sampling: [y_obs]


...

Sampling: [y_obs]


Output()

Sampling: [y_obs]


Output()

Built-in palettes: ['SAGE', 'CORPORATE', 'WARM', 'MONOCHROME', 'AUGUR']
Saved: artifacts/aurora_report.html (full) + artifacts/aurora_exec_summary.html (exec cut)


### Takeaways
- One call turns a fitted model into a **board-ready, themed HTML report** with uncertainty baked in.
- Sections are configurable; palettes are brandable; a **causal-assumptions** section makes the
  modelling honest to stakeholders.
- Last stop — tie the whole story into one decision: **`05_unified_workflow.ipynb`**.